In [12]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_excel("../muestra_terra/Terra_example.xlsx")

In [24]:
import re
from collections import Counter

# Asegúrate de tener cargado tu DataFrame como `df`
requests = df['Request'].dropna().astype(str).tolist()

# Patrones para detectar secciones dentro del texto
section_patterns = [
    r"in the ([A-Z\s]+?) section",
    r"in the ([A-Z\s]+?) page",
    r"on the ([A-Z\s]+?) section",
    r"on the ([A-Z\s]+?) page",
    r"in ([A-Z\s]+?) section",
    r"on ([A-Z\s]+?) page"
]

# Extraer y normalizar
found_sections = []
for text in requests:
    for pattern in section_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        found_sections.extend([m.strip().title() for m in matches])

# Contar ocurrencias
section_counts = Counter(found_sections)
print(section_counts.most_common(10))

[('Who We Are', 5), ('The Who We Are', 4), ('The', 1), ('Keeping The Header Copy The Same On Both', 1)]


In [ ]:
import pandas as pd
import random
from faker import Faker
from transformers import pipeline, set_seed
from datetime import date, datetime, timedelta

fake = Faker()
set_seed(42)

request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]

sections_base = ['header', 'footer', 'product page', 'sign-up form', 'testimonial section', 'pricing table', 'FAQ block']
modifiers = ['main', 'mobile version', 'desktop view', 'CTA zone', 'hover state', 'responsive layout']
sections = [f"{mod} of {base}" for base in sections_base for mod in random.sample(modifiers, 2)]

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

introductions = [
    "Hey team, this is {requester} from {company}.",
    "Hi there! {requester} from {company} here.",
    "{requester} here – I'm with {company}, working on {project}.",
    "Hi! I’m looking at {project} from {company}.",
    "Quick note from {requester} ({company}).",
    "{requester}, {company} – small issue I noticed.",
    "Hi all, reviewing {project}, found something.",
    "Heads up! {requester} from {company} noticed this.",
    "Reporting a detail in {project} – {requester}, {company}.",
]

tech_terms_by_type = {
    'Copy Revision': ['tone of voice', 'CTA', 'microcopy', 'UX writing'],
    'Design Issues': ['UI consistency', 'alignment', 'spacing', 'responsive layout', 'mobile-first'],
    'Requested Change': ['client feedback', 'scope update', 'functional change', 'new spec'],
    'New Item': ['feature', 'widget', 'module', 'custom component', 'new interaction']
}
buzzwords_common = ['UI', 'UX', 'layout', 'interaction', 'feedback', 'component', 'structure', 'copy', 'text', 'spacing']

generic_templates = [
    "Noticed something odd in {section}.",
    "Please check what's happening in {section}.",
    "Potential issue detected in {section}.",
    "Let’s improve {section}.",
    "Can we review the {section} again?"
]

templates = {
    'Copy Revision': [
        "Please revise the copy in {section}.",
        "Copy update needed for {section}.",
        "Text correction required in {section}.",
        "Rewriting needed for {section}.",
        "The content in {section} needs improvement."
    ],
    'Design Issues': [
        "Design problem found in {section}.",
        "Layout issue detected in {section}.",
        "Inconsistency in design of {section}.",
        "Visual bug in {section} needs fixing.",
        "Improve visual design in {section}."
    ],
    'Requested Change': [
        "Client requested a change in {section}.",
        "Requested modification for {section}.",
        "Adjustment needed in {section}.",
        "Update required following feedback in {section}.",
        "Changes needed according to client input in {section}."
    ],
    'New Item': [
        "New component needed in {section}.",
        "Add a new feature to {section}.",
        "Create something new in {section}.",
        "New item requested for {section}.",
        "Build a new section in {section}."
    ]
}

urgency_phrases = {
    'Low': ["", "No rush at all.", "Whenever there's time.", "Non-blocking."],
    'Normal': ["No big deal, but worth checking.", "Should go in this sprint.", "Minor but relevant."],
    'High': ["Pretty important, please prioritize.", "Client’s expecting this.", "Would prefer ASAP."],
    'Urgent': ["This broke things.", "Team is blocked.", "Immediate fix needed.", "On the critical path!"]
}

def generate_request(request_type, section, priority, company, requester, project, page):
    section_variants = [f"the {section}", f"section '{section}'", f"'{section}' section", f"area: {section}", f"in {section}"]
    section_phrase = random.choice(section_variants)

    if random.random() < 0.3:
        base_template = random.choice(generic_templates)
    else:
        base_template = random.choice(templates.get(request_type, ["Update {section}."]))

    base = base_template.replace("{section}", section_phrase)

    buzzword_pool = tech_terms_by_type.get(request_type, []) + buzzwords_common
    buzzword = random.choice(buzzword_pool) if buzzword_pool and random.random() < 0.5 else ""
    if buzzword:
        base += f" Consider {buzzword}."

    urgency = random.choice(urgency_phrases.get(priority, [""]))
    punctuation = random.choice([".", "!", "..."])

    intro = ""
    if random.random() < 0.7:
        intro_template = random.choice(introductions)
        intro = intro_template.format(
            requester=requester,
            company=company,
            project=project,
            page=page
        ) + " "

    return intro + base + urgency + punctuation

num_clients = 23
client_names = list({fake.company() for _ in range(num_clients * 2)})[:num_clients]
random.shuffle(client_names)
clients = {}
for i, name in enumerate(client_names):
    if i < 5:
        clients[name] = 'large'
    elif i < 15:
        clients[name] = 'medium'
    else:
        clients[name] = 'small'
size_weights = {'large': 5, 'medium': 3, 'small': 1}
weighted_clients = []
for client, size in clients.items():
    weighted_clients.extend([client] * size_weights[size])

project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
projects_per_client = {}
for client, size in clients.items():
    num_projects = {'large': random.randint(5, 8), 'medium': random.randint(3, 5), 'small': random.randint(1, 2)}[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        project_name = f"{base} {suffix} – {color}"
        projects.add(project_name)
    projects_per_client[client] = list(projects)

start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)
project_date_bases = {}
for project_list in projects_per_client.values():
    for project in project_list:
        project_date_bases[project] = fake.date_between(start_date=start_date, end_date=end_date)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

weighted_browsers = ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 + ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

data = []
for _ in range(10000):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)
    priority = random.choice(priority_weights_by_type[request_type])
    page = generate_client_page(project)
    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    real_time = max(1, round(random.normalvariate(estimated_time, 0.3)))
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]

    if status_value == "Complete":
        real_time = max(1, round(random.normalvariate(estimated_time, estimated_time * 0.3)))
    elif status_value == "In Progress":
        real_time = round(estimated_time * random.uniform(1.0, 1.5))
    else:
        real_time = 0

    requester = fake.name()

    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": requester,
        "Request Type": request_type,
        "Priority": priority,
        "Request": generate_request(request_type, section, priority, client, requester, project, page),
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })

df = pd.DataFrame(data)


In [51]:
client_attributes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23 entries, Airbnb to AXA
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Industry              23 non-null     int64  
 1   GrowthPotential       23 non-null     int64  
 2   CulturalFit           23 non-null     int64  
 3   Reputation            23 non-null     int64  
 4   Profitability         23 non-null     int64  
 5   LongTermRelationship  23 non-null     int64  
 6   ClientStrategicFit    23 non-null     float64
 7   Company               23 non-null     object 
dtypes: float64(1), int64(6), object(1)
memory usage: 2.2+ KB


In [72]:
client_attributes

,Industry,GrowthPotential,CulturalFit,Reputation,Profitability,LongTermRelationship,ClientStrategicFit,Company_f
Company,,,,,,,,
Airbnb,5,4,4,5,5,5,4.70,Wagner-Smith
UN,2,2,3,3,1,5,2.60,"Dickson, Clark and Green"
Abbott,4,4,4,4,4,5,4.15,House-Nguyen
Healthline Media,4,4,4,4,3,4,3.85,"Spencer, Pena and Cochran"
Talkdesk,4,5,4,4,2,4,3.90,"Rich, Thomas and Douglas"
Stifel Bank,3,4,3,3,4,4,3.50,"Combs, Gomez and Harris"
DAT,3,3,3,3,3,4,3.15,"Lucas, Madden and Hubbard"
Ruby's Rainbow,1,1,4,2,1,3,1.80,Hill-Pearson
SEI,4,5,4,4,4,4,4.20,Wells Group


In [ ]:
import pandas as pd
import random
from faker import Faker
from datetime import date, datetime, timedelta

# Inicialización
fake = Faker()

# Configuración de empresas con atributos
client_attributes = pd.read_csv("tiering_sample\client_strategic_fit.csv", index_col=0)  # Carga desde un CSV si lo usas externo

client_attributes['Company'] = [fake.company() for _ in range(len(client_attributes))]

# Configurar categorías
request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
sections = ['hero section', 'footer', 'about us page', 'contact form', 'pricing table', 'FAQ block']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

# Introducciones e ideas de lenguaje
introductions = [
    "Hey team, this is {requester} from {company}.",
    "Hi! {requester} here (from {company})",
    "Hey! Working on the {project}, I noticed the following:",
    "{requester} from {company}, reporting an issue on {page}",
    "Hi again! Quick one from {requester} at {company}.",
    "From {company} – {requester} noticed something in the UI.",
    "Hi, this is about the {project}. I’m {requester} from {company}."
]

tech_terms_by_type = {
    'Copy Revision': ['tone of voice', 'CTA', 'microcopy', 'UX writing'],
    'Design Issues': ['UI consistency', 'alignment', 'spacing', 'responsive layout', 'mobile-first'],
    'Requested Change': ['client feedback', 'scope update', 'functional change', 'new spec'],
    'New Item': ['feature', 'widget', 'module', 'custom component', 'new interaction']
}

def generate_request(request_type, section, priority, company, requester, project, page):
    section_variants = [
        f"the {section}", f"section '{section}'", f"'{section}' section",
        f"area: {section}", f"in {section}"
    ]

    urgency_phrases = {
        'Low': ["", " Not urgent.", " Can be handled later."],
        'Normal': [" Please address this soon.", " Handle this in the upcoming days.", " This should be scheduled accordingly."],
        'High': [" This is a high priority item.", " Should be prioritized.", " Requires prompt attention."],
        'Urgent': [" URGENT! This blocks progress.", " Immediate action required.", " Needs to be resolved ASAP."]
    }

    templates = {
        'Copy Revision': [
            "Please revise the copy in {section}.", "Copy update needed for {section}.",
            "Text correction required in {section}.", "Rewriting needed for {section}.",
            "The content in {section} needs improvement."
        ],
        'Design Issues': [
            "Design problem found in {section}.", "Layout issue detected in {section}.",
            "Inconsistency in design of {section}.", "Visual bug in {section} needs fixing.",
            "Improve visual design in {section}."
        ],
        'Requested Change': [
            "Client requested a change in {section}.", "Requested modification for {section}.",
            "Adjustment needed in {section}.", "Update required following feedback in {section}.",
            "Changes needed according to client input in {section}."
        ],
        'New Item': [
            "New component needed in {section}.", "Add a new feature to {section}.",
            "Create something new in {section}.", "New item requested for {section}.",
            "Build a new section in {section}."
        ]
    }

    tech_terms = tech_terms_by_type.get(request_type, [])
    buzzword = random.choice(tech_terms) if tech_terms and random.random() < 0.5 else ""
    section_phrase = random.choice(section_variants)
    base_template = random.choice(templates.get(request_type, ["Update {section}."]))
    base = base_template.replace("{section}", section_phrase)
    if buzzword:
        base += f" Consider {buzzword}."
    urgency = random.choice(urgency_phrases.get(priority, [""]))

    intro = ""
    if random.random() < 0.7:
        intro_template = random.choice(introductions)
        intro = intro_template.format(
            requester=requester, company=company, project=project, page=page
        ) + " "

    return intro + base + urgency

# Estimación de tiempo por tipo
estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

# Generador de página web
def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

# Browser y device
weighted_browsers = ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 + ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

# Fechas base por proyecto
start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

# Lógica por cliente
project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
project_date_bases = {}
projects_per_client = {}
weighted_clients = []

for _, row in client_attributes.iterrows():
    company = row['Company']
    strategic_fit = row['ClientStrategicFit']
    size = 'large' if strategic_fit >= 4.5 else 'medium' if strategic_fit >= 3.0 else 'small'
    num_projects = {
        'large': random.randint(5, 8),
        'medium': random.randint(3, 5),
        'small': random.randint(1, 2)
    }[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        projects.add(f"{base} {suffix} – {color}")
    projects = list(projects)
    projects_per_client[company] = projects
    for p in projects:
        project_date_bases[p] = fake.date_between(start_date=start_date, end_date=end_date)
    weighted_clients.extend([company] * int(strategic_fit * 2))

# Generación de datos
n = 10000
data = []
for _ in range(n):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)
    priority = random.choice(priority_weights_by_type[request_type])
    page = generate_client_page(project)

    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]

    if status_value == "Complete":
        real_time = max(1, round(random.normalvariate(estimated_time, estimated_time * 0.3)))
    elif status_value == "In Progress":
        real_time = round(estimated_time * random.uniform(1.0, 1.5))
    else:
        real_time = 0

    requester = fake.name()
    request = generate_request(request_type, section, priority, client, requester, project, page)

    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": requester,
        "Request Type": request_type,
        "Priority": priority,
        "Request": request,
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })


# Convertir a DataFrame
df = pd.DataFrame(data)

# Añadir el strategic
mapping = client_attributes.set_index('Company')['ClientStrategicFit']
df['ClientStrategicFit'] = df['Company'].map(mapping)


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Company                  10000 non-null  object
 1   Project Name             10000 non-null  object
 2   Input Date               10000 non-null  object
 3   Status                   10000 non-null  object
 4   Requester                10000 non-null  object
 5   Request Type             10000 non-null  object
 6   Priority                 10000 non-null  object
 7   Request                  10000 non-null  object
 8   Device                   10000 non-null  object
 9   Browser                  10000 non-null  object
 10  Page                     10000 non-null  object
 11  Estimated Time (tokens)  10000 non-null  int64 
 12  Real Time                10000 non-null  int64 
dtypes: int64(2), object(11)
memory usage: 1015.8+ KB


In [13]:
df = pd.read_csv("synthetic_example.csv")

In [73]:
df

,Company,Project Name,Input Date,Status,Requester,Request Type,Priority,Request,Device,Browser,Page,Estimated Time (tokens),Real Time
0,"Rich, Thomas and Douglas",Call Dashboard – MediumSlateBlue,09/04/2020,Complete,Joshua Brennan,Requested Change,Urgent,Hey! Working on the Call Dashboard – MediumSla...,Desktop,Safari,https://www.calldashboard–mediumslateblue.com,5,6
1,"Spencer, Pena and Cochran",Service Dashboard – Lavender,13/10/2024,Complete,Christian Owen,Requested Change,Normal,Update required following feedback in in FAQ b...,Mobile,Chrome,https://www.servicedashboard–lavender.com,13,9
2,Cobb Ltd,Teach Redesign – OliveDrab,12/06/2022,Complete,James Hughes,Design Issues,High,Layout issue detected in area: contact form. C...,Desktop,Safari,https://www.teachredesign–olivedrab.com,3,4
3,"Wright, Kaiser and Smith",Someone Platform – Navy,15/09/2022,Complete,Rick Schroeder,Copy Revision,Normal,"Hi, this is about the Someone Platform – Navy....",Desktop,Chrome,https://www.someoneplatform–navy.com,1,1
4,Wells Group,Decade Landing – SandyBrown,26/11/2022,Complete,Donna Oneill,Copy Revision,Normal,"Hey team, this is Donna Oneill from Wells Grou...",Desktop,Chrome,https://www.decadelanding–sandybrown.com,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Parker-Daniels,Ball Revamp – Salmon,18/03/2024,Complete,Eric Farrell,New Item,Urgent,"Hey! Working on the Ball Revamp – Salmon, I no...",Mobile,Firefox,https://www.ballrevamp–salmon.com,8,11
9996,"Dickson, Clark and Green",Else Redesign – NavajoWhite,19/06/2022,Complete,Charles Bell,Design Issues,Low,Hi again! Quick one from Charles Bell at Dicks...,Tablet,Safari,https://www.elseredesign–navajowhite.com,3,5
9997,"Pace, David and Vargas",Decade Portal – CornflowerBlue,31/01/2024,Complete,Donna May,New Item,High,New item requested for 'hero section' section....,Mobile,Chrome,https://www.decadeportal–cornflowerblue.com,13,18
9998,"Rich, Thomas and Douglas",Call Dashboard – MediumSlateBlue,17/02/2020,Complete,Karen Watkins,Copy Revision,Low,Text correction required in in contact form. C...,Tablet,Chrome,https://www.calldashboard–mediumslateblue.com,1,1


In [34]:
# Número de proyectos por empresa
proyectos_por_empresa = df[['Company', 'Project Name']].drop_duplicates()
proyectos_count = proyectos_por_empresa.groupby('Company').size().reset_index(name='Num Projects')

# Número de requests por proyecto
requests_count = df.groupby(['Company', 'Project Name']).size().reset_index(name='Num Requests')

# Merge para unir ambos
combined = requests_count.merge(proyectos_count, on='Company')

# Mostrar ordenado por empresa y luego por cantidad de requests
combined.sort_values(['Num Projects', 'Num Requests'], ascending=[False, False])

,Company,Project Name,Num Requests,Num Projects
46,"Mack, Smith and Ingram",Sort Platform – Wheat,90,8
40,"Mack, Smith and Ingram",Example Platform – MidnightBlue,87,8
45,"Mack, Smith and Ingram",Several Dashboard – Aqua,82,8
43,"Mack, Smith and Ingram",Section Campaign – DarkGray,78,8
44,"Mack, Smith and Ingram",Sell Dashboard – DarkOrange,74,8
...,...,...,...,...
80,Tran-Lewis,Cold Revamp – MediumSpringGreen,103,2
0,"Bates, Jackson and Green",Rule Revamp – MediumPurple,369,1
24,Harris-Farmer,Product Landing – Aqua,333,1
69,Perry-Gibson,Alone Portal – RosyBrown,275,1


In [35]:
#df2 = df[["Request Type" , "Status", "Input Date", "Requester", "Device", "Browser", "Request", "Page"]]

df.to_csv("synthetic_example.csv", index=False)

df.to_json("synthetic_example.json", orient="records", lines=True)